# Full Factorial Design Analysis: Adaptive Caliper Selection

## Comprehensive Simulation Study Across 54 Scenarios

**Authors:** Perkins Watambwa et al.  
**Institution:** Centre for Sexual Health and HIV/AIDS Research (CeSHHAR), Zimbabwe

---

This notebook runs and analyzes the complete factorial design simulation:

**Simulation Design Factors:**
- **Sample size (n):** 500, 1000, 2000
- **Treatment prevalence:** 0.3, 0.5, 0.7
- **Overlap (coefficient magnitude):** Low (α = 0.3), Medium (α = 0.6), High (α = 1.0)
- **Confounding strength:** Weak (β = 0.3), Strong (β = 1.0)

**Total scenarios:** 3 × 3 × 3 × 2 = 54

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from config import *
from simulation_runner import run_full_simulation, summarize_results
from visualization import (
    plot_factorial_heatmaps,
    plot_method_ranking_across_scenarios,
    create_comprehensive_comparison_table,
    plot_scenario_factor_effects,
    create_scenario_comparison_figure,
    plot_method_comparison_boxplot
)

plt.rcParams.update(PLT_PARAMS)
sns.set_style("whitegrid")

print(f"Output directory: {OUTPUT_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"Tables directory: {TABLES_DIR}")
print(f"Results directory: {RESULTS_DIR}")

Output directory: /Users/perkins/Desktop/CeSHHAR/HAPI/Adaptive Caliper Simulations/notebooks/../outputs
Figures directory: /Users/perkins/Desktop/CeSHHAR/HAPI/Adaptive Caliper Simulations/notebooks/../outputs/figures
Tables directory: /Users/perkins/Desktop/CeSHHAR/HAPI/Adaptive Caliper Simulations/notebooks/../outputs/tables
Results directory: /Users/perkins/Desktop/CeSHHAR/HAPI/Adaptive Caliper Simulations/notebooks/../outputs/results


## 1. Generate All Scenarios

First, let's generate all 54 scenarios from the factorial design.

In [ ]:
# Generate all scenarios
scenarios = generate_all_scenarios()

print(f"Total scenarios: {len(scenarios)}")
print("\nFactorial Design:")
print(f"  Sample sizes: {SAMPLE_SIZES}")
print(f"  Treatment prevalences: {TREATMENT_PREVALENCES}")
print(f"  Overlap levels: {list(OVERLAP_LEVELS.keys())}")
print(f"  Confounding strengths: {list(CONFOUNDING_STRENGTHS.keys())}")
print(f"\nReplications per scenario: {N_REPLICATIONS}")
print(f"Total simulation runs: {len(scenarios) * N_REPLICATIONS:,}")

In [ ]:
# Display first 10 scenarios
scenarios_df = pd.DataFrame(scenarios)
print("\nFirst 10 Scenarios:")
scenarios_df.head(10)

In [ ]:
# Summary of factorial design
print("\nScenario Distribution:")
print("\nBy Sample Size:")
print(scenarios_df['n'].value_counts().sort_index())
print("\nBy Treatment Prevalence:")
print(scenarios_df['treatment_prevalence'].value_counts().sort_index())
print("\nBy Overlap Level:")
print(scenarios_df['overlap_name'].value_counts())
print("\nBy Confounding Strength:")
print(scenarios_df['confounding_name'].value_counts())

## 2. Run Full Simulation

**Note:** This will take considerable time depending on your system.
- With parallelization on 8 cores: ~30-60 minutes
- Without parallelization: several hours

The simulation will save intermediate results automatically.

In [ ]:
# Option 1: Run full simulation (uncomment to run)
# WARNING: This takes a long time!

# results_df = run_full_simulation(
#     scenarios=scenarios,
#     n_replications=N_REPLICATIONS,
#     n_jobs=-1,  # Use all cores
#     save_intermediate=True,
#     verbose=True
# )

In [ ]:
# Option 2: Run a subset for quick testing (e.g., first 6 scenarios)
test_scenarios = scenarios[:6]  # Test with 6 scenarios
test_replications = 50  # Use fewer replications for testing

print(f"Running test simulation with {len(test_scenarios)} scenarios and {test_replications} replications...")

results_df = run_full_simulation(
    scenarios=test_scenarios,
    n_replications=test_replications,
    n_jobs=-1,
    save_intermediate=True,
    verbose=True
)

In [ ]:
# Option 3: Load previously saved results
# results_df = pd.read_csv(RESULTS_DIR / 'simulation_results_full.csv')
# print(f"Loaded {len(results_df)} simulation results")

In [ ]:
# Examine raw results
print(f"Results shape: {results_df.shape}")
print(f"\nColumns: {results_df.columns.tolist()}")
print(f"\nMethods evaluated: {results_df['method'].unique()}")
print(f"\nScenarios completed: {results_df['scenario_id'].nunique()}")

results_df.head()

## 3. Summarize Results

Aggregate results across replications for each scenario and method.

In [ ]:
# Generate summary statistics
summary_df = summarize_results(results_df)

print(f"Summary shape: {summary_df.shape}")
print(f"\nSummary columns:")
print(summary_df.columns.tolist())

summary_df.head(10)

In [ ]:
# Save summary
summary_df.to_csv(TABLES_DIR / 'simulation_summary_all_scenarios.csv', index=False)
print(f"Summary saved to {TABLES_DIR / 'simulation_summary_all_scenarios.csv'}")

## 4. Overall Performance Comparison

Compare methods averaged across all scenarios.

In [ ]:
# Overall performance table
overall_comparison = create_comprehensive_comparison_table(
    summary_df,
    save_path=TABLES_DIR / 'comprehensive_comparison_all_scenarios.csv'
)

print("\n" + "="*80)
print("OVERALL PERFORMANCE ACROSS ALL SCENARIOS")
print("="*80)
overall_comparison

In [ ]:
# Method performance by key metrics
method_summary = summary_df.groupby('method').agg({
    'mean_retention': ['mean', 'std', 'min', 'max'],
    'mean_max_smd': ['mean', 'std', 'min', 'max'],
    'mean_abs_bias': ['mean', 'std', 'min', 'max'],
    'rmse': ['mean', 'std', 'min', 'max'],
    'coverage_rate': ['mean', 'std', 'min', 'max']
}).round(4)

print("\nDetailed Method Summary (Mean ± SD [Min, Max]):")
method_summary

## 5. Visualization: Method Comparison Boxplots

Compare distributions of performance metrics across all scenarios.

In [ ]:
# Boxplot for bias
fig = plot_method_comparison_boxplot(
    results_df,
    metric='bias',
    title='Treatment Effect Bias Across All Scenarios',
    figsize=(12, 6)
)
plt.show()

In [ ]:
# Boxplot for MSE
fig = plot_method_comparison_boxplot(
    results_df,
    metric='mse',
    title='Mean Squared Error Across All Scenarios',
    figsize=(12, 6)
)
plt.show()

In [ ]:
# Boxplot for balance (max |SMD|)
fig = plot_method_comparison_boxplot(
    results_df,
    metric='max_smd',
    title='Covariate Balance (Max |SMD|) Across All Scenarios',
    figsize=(12, 6)
)
plt.show()

In [ ]:
# Boxplot for retention
fig = plot_method_comparison_boxplot(
    results_df,
    metric='retention',
    title='Sample Retention Across All Scenarios',
    figsize=(12, 6)
)
plt.show()

In [ ]:
# Boxplot for coverage
fig = plot_method_comparison_boxplot(
    results_df,
    metric='coverage',
    title='Coverage Probability Across All Scenarios',
    figsize=(12, 6)
)
plt.show()

## 6. Method Rankings

Analyze which methods perform best across scenarios.

In [ ]:
# Method rankings by RMSE
fig = plot_method_ranking_across_scenarios(
    summary_df,
    metric='rmse',
    figsize=(14, 6)
)
plt.show()

In [ ]:
# Method rankings by balance
fig = plot_method_ranking_across_scenarios(
    summary_df,
    metric='mean_max_smd',
    figsize=(14, 6)
)
plt.show()

In [ ]:
# Win rate analysis
metrics = ['rmse', 'mean_abs_bias', 'mean_max_smd', 'coverage_rate']
metric_labels = ['RMSE', 'Absolute Bias', 'Max |SMD|', 'Coverage Rate']

win_rate_data = []
for metric in metrics:
    best_method = summary_df.loc[summary_df.groupby('scenario_id')[metric].idxmin()]
    win_counts = best_method['method'].value_counts()
    win_rate_data.append(win_counts)

win_rate_df = pd.DataFrame(win_rate_data, index=metric_labels).fillna(0)
win_rate_pct = (win_rate_df / summary_df['scenario_id'].nunique() * 100).round(1)

print("\nWin Rate (% of scenarios where method is best):")
print("="*80)
win_rate_pct

## 7. Factorial Design Heatmaps

Visualize performance across factorial design factors.

In [ ]:
# Generate all heatmaps
heatmap_figures = plot_factorial_heatmaps(
    summary_df,
    metrics=['rmse', 'mean_abs_bias', 'coverage_rate', 'mean_retention', 'mean_max_smd'],
    figsize=(20, 4)
)

# Display RMSE heatmap
plt.figure(heatmap_figures['rmse'].number)
plt.show()

In [ ]:
# Display balance heatmap
plt.figure(heatmap_figures['mean_max_smd'].number)
plt.show()

In [ ]:
# Display coverage heatmap
plt.figure(heatmap_figures['coverage_rate'].number)
plt.show()

In [ ]:
# Display retention heatmap
plt.figure(heatmap_figures['mean_retention'].number)
plt.show()

## 8. Factorial Design Factor Effects

Examine main effects of each design factor.

In [ ]:
# Factor effects on RMSE
fig = plot_scenario_factor_effects(
    summary_df,
    metric='rmse',
    figsize=(14, 10)
)
plt.show()

In [ ]:
# Factor effects on balance
fig = plot_scenario_factor_effects(
    summary_df,
    metric='mean_max_smd',
    figsize=(14, 10)
)
plt.show()

In [ ]:
# Factor effects on coverage
fig = plot_scenario_factor_effects(
    summary_df,
    metric='coverage_rate',
    figsize=(14, 10)
)
plt.show()

## 9. Scenario-by-Scenario Comparison

Track performance across all scenarios.

In [ ]:
# Multi-panel scenario comparison
fig = create_scenario_comparison_figure(
    summary_df,
    figsize=(16, 10)
)
plt.show()

## 10. Detailed Analysis by Factorial Design Factors

In [ ]:
# Performance by sample size
print("\nPerformance by Sample Size:")
print("="*80)
by_n = summary_df.groupby(['n', 'method']).agg({
    'rmse': 'mean',
    'mean_abs_bias': 'mean',
    'mean_max_smd': 'mean',
    'coverage_rate': 'mean',
    'mean_retention': 'mean'
}).round(4)
by_n

In [ ]:
# Performance by overlap level
print("\nPerformance by Overlap Level:")
print("="*80)
by_overlap = summary_df.groupby(['overlap_name', 'method']).agg({
    'rmse': 'mean',
    'mean_abs_bias': 'mean',
    'mean_max_smd': 'mean',
    'coverage_rate': 'mean',
    'mean_retention': 'mean'
}).round(4)
by_overlap

In [ ]:
# Performance by confounding strength
print("\nPerformance by Confounding Strength:")
print("="*80)
by_conf = summary_df.groupby(['confounding_name', 'method']).agg({
    'rmse': 'mean',
    'mean_abs_bias': 'mean',
    'mean_max_smd': 'mean',
    'coverage_rate': 'mean',
    'mean_retention': 'mean'
}).round(4)
by_conf

In [ ]:
# Performance by treatment prevalence
print("\nPerformance by Treatment Prevalence:")
print("="*80)
by_prev = summary_df.groupby(['treatment_prevalence', 'method']).agg({
    'rmse': 'mean',
    'mean_abs_bias': 'mean',
    'mean_max_smd': 'mean',
    'coverage_rate': 'mean',
    'mean_retention': 'mean'
}).round(4)
by_prev

## 11. Identify Best and Worst Scenarios for Each Method

In [ ]:
# For each method, find best and worst performing scenarios
for method in ['ACS-Balance', 'ACS-Knee', 'ACS-Weighted', 'Fixed-0.2']:
    method_data = summary_df[summary_df['method'] == method].copy()
    
    best_idx = method_data['rmse'].idxmin()
    worst_idx = method_data['rmse'].idxmax()
    
    best = method_data.loc[best_idx]
    worst = method_data.loc[worst_idx]
    
    print(f"\n{method}:")
    print("-" * 60)
    print(f"  Best scenario (lowest RMSE): Scenario {best['scenario_id']}")
    print(f"    n={best['n']}, prev={best['treatment_prevalence']}, overlap={best['overlap_name']}, conf={best['confounding_name']}")
    print(f"    RMSE={best['rmse']:.4f}, Balance={best['mean_max_smd']:.4f}, Retention={best['mean_retention']:.2%}")
    
    print(f"\n  Worst scenario (highest RMSE): Scenario {worst['scenario_id']}")
    print(f"    n={worst['n']}, prev={worst['treatment_prevalence']}, overlap={worst['overlap_name']}, conf={worst['confounding_name']}")
    print(f"    RMSE={worst['rmse']:.4f}, Balance={worst['mean_max_smd']:.4f}, Retention={worst['mean_retention']:.2%}")

## 12. Export Final Results

In [ ]:
# Save all key tables
print("Exporting results...")

# Overall summary
overall_comparison.to_csv(TABLES_DIR / 'table1_overall_comparison.csv')

# By factor summaries
by_n.to_csv(TABLES_DIR / 'table2_by_sample_size.csv')
by_overlap.to_csv(TABLES_DIR / 'table3_by_overlap.csv')
by_conf.to_csv(TABLES_DIR / 'table4_by_confounding.csv')
by_prev.to_csv(TABLES_DIR / 'table5_by_prevalence.csv')

# Win rates
win_rate_pct.to_csv(TABLES_DIR / 'table6_win_rates.csv')

# Save all figures
for metric, fig in heatmap_figures.items():
    fig.savefig(FIGURES_DIR / f'figure_heatmap_{metric}.png', dpi=300, bbox_inches='tight')
    plt.close(fig)

print("\nAll results exported!")
print(f"  Tables: {TABLES_DIR}")
print(f"  Figures: {FIGURES_DIR}")

## Summary

This notebook has analyzed the complete factorial design simulation across all 54 scenarios, comparing:
- Fixed-caliper methods (0.1, 0.2, 0.5 SD)
- No caliper matching
- Adaptive Caliper Selection (ACS) methods: Balance, Knee, and Weighted

Key outputs:
1. **Comprehensive comparison tables** showing overall and factor-specific performance
2. **Method ranking visualizations** identifying best-performing approaches
3. **Factorial design heatmaps** showing performance across all factor combinations
4. **Factor effect plots** examining main effects of design factors
5. **Boxplots and distribution comparisons** for all key metrics

All results are saved to the outputs directory for manuscript preparation.